In [ ]:
# ============================================================
# Imports
# ============================================================
import os
import random
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.signal import savgol_filter

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf

# ============================================================
# Parameters
# ============================================================
data_dir = "/content/drive/MyDrive/"

time_steps = 5
smoothing_window = 5
smoothing_poly = 3
pca_components = 80

seeds = [42]
epochs = 50
batch_size = 32

output_csv = os.path.join(data_dir, "final_model_comparison_full.csv")

# ============================================================
# Seed Control
# ============================================================
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

# ============================================================
# Metrics
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

# ============================================================
# Load Data
# ============================================================
train_df = pd.read_csv(os.path.join(data_dir, "WO_rssi_theta_timeseries_Ch1_train.csv"), header=None)
test_df  = pd.read_csv(os.path.join(data_dir, "WO_rssi_theta_timeseries_Ch1_test.csv"), header=None)

X_train_raw = train_df.iloc[1:, :-1].values
y_train_raw = train_df.iloc[1:, -1].values
X_test_raw  = test_df.iloc[1:, :-1].values
y_test_raw  = test_df.iloc[1:, -1].values

# ============================================================
# Filter Classes
# ============================================================
selected_labels = ["P1", "P2", "P3", "P4"]
train_mask = np.isin(y_train_raw, selected_labels)
test_mask  = np.isin(y_test_raw, selected_labels)

X_train_raw, y_train_raw = X_train_raw[train_mask], y_train_raw[train_mask]
X_test_raw,  y_test_raw  = X_test_raw[test_mask],  y_test_raw[test_mask]

# ============================================================
# Encode Labels
# ============================================================
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)
num_classes = len(np.unique(y_train))

# ============================================================
# Scaling + Smoothing + Imputation
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

for i in range(X_train_scaled.shape[1]):
    X_train_scaled[:, i] = savgol_filter(X_train_scaled[:, i], smoothing_window, smoothing_poly, mode="nearest")
    X_test_scaled[:, i]  = savgol_filter(X_test_scaled[:, i], smoothing_window, smoothing_poly, mode="nearest")

imputer = SimpleImputer(strategy="mean")
X_train_scaled = imputer.fit_transform(X_train_scaled)
X_test_scaled  = imputer.transform(X_test_scaled)

# ============================================================
# Time-Domain Feature Extraction (ML & DNN)
# ============================================================
def time_domain_features(X, time_steps):
    num_samples, num_features = X.shape
    num_subcarriers = num_features // time_steps
    out = []
    for i in range(num_samples):
        feats = []
        for j in range(num_subcarriers):
            seg = X[i, j*time_steps:(j+1)*time_steps]
            feats.extend([seg.mean(), seg.std(), seg.min(), seg.max()])
        out.append(feats)
    return np.array(out)

X_train_td = time_domain_features(X_train_scaled, time_steps)
X_test_td  = time_domain_features(X_test_scaled, time_steps)

# ============================================================
# PCA
# ============================================================
pca = PCA(n_components=pca_components)
X_train_pca = pca.fit_transform(X_train_td)
X_test_pca  = pca.transform(X_test_td)

# ============================================================
# ML Models
# ============================================================
ml_models = {
    "DecisionTree": DecisionTreeClassifier(max_depth=20),
    "RandomForest": RandomForestClassifier(n_estimators=200),
    "SVM_RBF": SVC(kernel="rbf"),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "MLP": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=50,
        learning_rate=0.01,
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss"
    )
}

results = []

# ============================================================
# ML — With Feature Extraction
# ============================================================
for name, model in ml_models.items():
    mets = []
    for seed in seeds:
        set_seed(seed)
        model.fit(X_train_pca, y_train)
        mets.append(compute_metrics(y_test, model.predict(X_test_pca)))
    results.append({"Model": name, "Setting": "With Feature Extraction", **{k: np.mean([m[k] for m in mets]) for k in mets[0]}})

# ============================================================
# ML — Raw Signals
# ============================================================
for name, model in ml_models.items():
    mets = []
    for seed in seeds:
        set_seed(seed)
        model.fit(X_train_scaled, y_train)
        mets.append(compute_metrics(y_test, model.predict(X_test_scaled)))
    results.append({"Model": name, "Setting": "Raw Signals Only", **{k: np.mean([m[k] for m in mets]) for k in mets[0]}})

# ============================================================
# Prepare DL Raw Signals
# ============================================================
num_features = X_train_scaled.shape[1] // time_steps
X_train_dl = X_train_scaled.reshape(-1, time_steps, num_features)
X_test_dl  = X_test_scaled.reshape(-1, time_steps, num_features)

# ============================================================
# DL Builders
# ============================================================
def build_dnn(shape):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=shape),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

def build_cnn1d(shape):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=shape),
        tf.keras.layers.Conv1D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

def build_rnn(shape):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=shape),
        tf.keras.layers.SimpleRNN(64),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

def build_lstm(shape):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=shape),
        tf.keras.layers.LSTM(64),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

dl_models = {
    "DNN": build_dnn,
    "CNN1D": build_cnn1d,
    "RNN": build_rnn,
    "LSTM": build_lstm
}

# ============================================================
# DL — Raw Signals
# ============================================================
for name, builder in dl_models.items():
    mets = []
    for seed in seeds:
        set_seed(seed)
        model = builder(X_train_dl.shape[1:])
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
        model.fit(X_train_dl, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        y_pred = np.argmax(model.predict(X_test_dl, verbose=0), axis=1)
        mets.append(compute_metrics(y_test, y_pred))
    results.append({"Model": name, "Setting": "Raw Signals Only", **{k: np.mean([m[k] for m in mets]) for k in mets[0]}})

# ============================================================
# DNN — With Feature Extraction
# ============================================================
mets = []
for seed in seeds:
    set_seed(seed)
    model = build_dnn(X_train_pca.shape[1:])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    model.fit(X_train_pca, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
    y_pred = np.argmax(model.predict(X_test_pca, verbose=0), axis=1)
    mets.append(compute_metrics(y_test, y_pred))

results.append({"Model": "DNN", "Setting": "With Feature Extraction", **{k: np.mean([m[k] for m in mets]) for k in mets[0]}})

# ============================================================
# Save Results
# ============================================================
final_df = pd.DataFrame(results)
final_df.to_csv(output_csv, index=False)

print("\n✅ FINAL RESULTS (sorted by F1-score)\n")
print(final_df.sort_values("F1", ascending=False))


In [ ]:

✅ FINAL RESULTS (sorted by F1-score)

                 Model                  Setting  Accuracy  Precision  \
12                 MLP         Raw Signals Only  0.715313   0.667100
14                 DNN         Raw Signals Only  0.713156   0.667360
11  LogisticRegression         Raw Signals Only  0.707405   0.657869
15               CNN1D         Raw Signals Only  0.698778   0.654237
9              SVM_RBF         Raw Signals Only  0.713156   0.657557
16                 RNN         Raw Signals Only  0.695183   0.640259
13             XGBoost         Raw Signals Only  0.693027   0.629123
17                LSTM         Raw Signals Only  0.658519   0.607302
2              SVM_RBF  With Feature Extraction  0.667146   0.584491
4   LogisticRegression  With Feature Extraction  0.647735   0.569849
5                  MLP  With Feature Extraction  0.601006   0.542792
8         RandomForest         Raw Signals Only  0.682243   0.618758
18                 DNN  With Feature Extraction  0.586628   0.533330
6              XGBoost  With Feature Extraction  0.624730   0.543063
3                  KNN  With Feature Extraction  0.613947   0.549985
10                 KNN         Raw Signals Only  0.616104   0.526009
0         DecisionTree  With Feature Extraction  0.549245   0.511548
7         DecisionTree         Raw Signals Only  0.532710   0.488607
1         RandomForest  With Feature Extraction  0.621136   0.531799

      Recall        F1
12  0.665907  0.666124
14  0.663053  0.664740
11  0.661178  0.658803
15  0.651035  0.652552
9   0.648710  0.645040
16  0.649052  0.643438
13  0.617265  0.615991
17  0.608258  0.607693
2   0.582827  0.566056
4   0.578677  0.565987
5   0.551152  0.546281
8   0.569511  0.544538
18  0.535847  0.534441
6   0.543939  0.529223
3   0.528247  0.527305
10  0.526399  0.512723
0   0.511804  0.511468
7   0.489376  0.488764
1   0.497606  0.472031